In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import time
import json
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports ready")

✓ Imports ready


In [ ]:
FEATURES = [
    'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR', 'IS_HOLIDAY',
    'DISTANCE', 'profit_margin', 'origin_monthly_passengers',
    'origin_temp', 'origin_dew_point', 'origin_pressure',
    'origin_wind_dir', 'origin_wind_speed', 'origin_precip_1hr',
    'origin_weather_severity',
    'dest_temp', 'dest_dew_point', 'dest_pressure',
    'dest_wind_dir', 'dest_wind_speed', 'dest_precip_1hr',
    'dest_weather_severity',
    'airline_delay_rate_30d', 'origin_delay_rate_30d', 'dest_delay_rate_30d',
    'route_delay_rate_30d', 'origin_avg_taxi_out_30d',
    'flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d',
    'carrier_origin_delay_rate_30d', 'dest_hour_delay_rate_30d',
    'airline_delay_rate_7d', 'origin_delay_rate_7d',
    'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay',
    'hourly_flight_count', 'scheduled_turnaround_buffer', 'tail_flight_num_today',
    'dest_hourly_flight_count',
    'inbound_arr_delay_3h', 'dest_inbound_arr_delay_3h',
    'prev_tail_arr_delay', 'national_hub_delay_2h',
    'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label',
    'origin_pressure_delta_3h', 'dest_pressure_delta_3h',
    'origin_wind_speed_delta_3h', 'dest_wind_speed_delta_3h',
    'day_of_year',
    'origin_dep_delay_rate_1h', 'dest_dep_delay_rate_1h',
    'origin_stress_index', 'real_time_turn_gap',
    'tail_delays_today', 'tail_active_hours',
    'origin_pressure_drop_stress', 'airport_fatigue_index',
]
CAT_FEATURES = ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label']
TARGET = 'ARR_DELAY'

flights = pd.read_parquet('dataset/merged_flights_fe_v2.parquet')
for col in CAT_FEATURES:
    flights[col] = flights[col].astype('category')

print(f"ARR_DELAY stats:")
print(f"  Nulls: {flights[TARGET].isna().sum():,} ({flights[TARGET].isna().mean()*100:.1f}%)")
print(f"  Mean:  {flights[TARGET].mean():.1f} min")
print(f"  Median: {flights[TARGET].median():.1f} min")
print(f"  Std:   {flights[TARGET].std():.1f} min")
print(f"  Min:   {flights[TARGET].min():.1f} min")
print(f"  Max:   {flights[TARGET].max():.1f} min")

flights = flights.dropna(subset=[TARGET])

train = flights[flights['FL_DATE'] < '2025-01-01']
test  = flights[flights['FL_DATE'] >= '2025-01-01']

X_train = train[FEATURES].copy()
y_train = train[TARGET].copy()
X_test  = test[FEATURES].copy()
y_test  = test[TARGET].copy()

del flights, train, test

print(f"\nFeatures: {len(FEATURES)}")
print(f"Train: {X_train.shape[0]:,}  |  Mean delay: {y_train.mean():.1f} min")
print(f"Test:  {X_test.shape[0]:,}  |  Mean delay: {y_test.mean():.1f} min")

ARR_DELAY stats:
  Nulls: 0 (0.0%)
  Mean:  7.5 min
  Median: -6.0 min
  Std:   58.5 min
  Min:   -128.0 min
  Max:   4405.0 min

Features: 61
Train: 13,708,670  |  Mean delay: 6.9 min
Test:  4,519,126  |  Mean delay: 9.4 min


In [3]:
CAP_HIGH = 300
CAP_LOW = -60
y_train_capped = y_train.clip(lower=CAP_LOW, upper=CAP_HIGH)
y_test_capped  = y_test.clip(lower=CAP_LOW, upper=CAP_HIGH)

model_reg = lgb.LGBMRegressor(
    objective='huber', metric='mae', verbose=-1,
    random_state=42, n_jobs=-1,
    n_estimators=20000,
    learning_rate=0.0990242280231697,
    num_leaves=333,
    max_depth=12,
    min_child_samples=301,
    subsample=0.8013995149182327,
    colsample_bytree=0.9758270708358225,
    reg_alpha=4.72656018100241,
    reg_lambda=0.05637038386333235,
    min_split_gain=0.4461068357921353,
    max_bin=210,
)

print("Training regression (Huber + capped, 61 features)...")
start = time.time()
model_reg.fit(
    X_train, y_train_capped,
    eval_set=[(X_test, y_test_capped)],
    callbacks=[lgb.early_stopping(200), lgb.log_evaluation(500)],
)
elapsed = time.time() - start
print(f"Done: {elapsed/60:.1f} min | Iters: {model_reg.best_iteration_}")

import joblib
joblib.dump(model_reg, 'models/lgbm_delay_regressor_final.pkl')
print("✓ MODEL SAVED")

Training regression (Huber + capped, 61 features)...
Training until validation scores don't improve for 200 rounds
[500]	valid_0's l1: 19.738
[1000]	valid_0's l1: 18.4118
[1500]	valid_0's l1: 17.7776
[2000]	valid_0's l1: 17.4562
[2500]	valid_0's l1: 17.2745
[3000]	valid_0's l1: 17.1676
[3500]	valid_0's l1: 17.0949
[4000]	valid_0's l1: 17.0461
[4500]	valid_0's l1: 17.0111
[5000]	valid_0's l1: 16.9873
[5500]	valid_0's l1: 16.9665
[6000]	valid_0's l1: 16.9508
[6500]	valid_0's l1: 16.9366
[7000]	valid_0's l1: 16.9245
[7500]	valid_0's l1: 16.9133
[8000]	valid_0's l1: 16.9028
[8500]	valid_0's l1: 16.8947
[9000]	valid_0's l1: 16.8871
[9500]	valid_0's l1: 16.8809
[10000]	valid_0's l1: 16.8748
[10500]	valid_0's l1: 16.8679
[11000]	valid_0's l1: 16.8639
[11500]	valid_0's l1: 16.8609
[12000]	valid_0's l1: 16.858
[12500]	valid_0's l1: 16.8549
[13000]	valid_0's l1: 16.8512
[13500]	valid_0's l1: 16.8486
[14000]	valid_0's l1: 16.8458
[14500]	valid_0's l1: 16.8436
[15000]	valid_0's l1: 16.842
[15500]	

In [4]:
y_pred = model_reg.predict(X_test)
print(f"Predictions: {len(y_pred):,}")

Predictions: 4,519,126


In [5]:
mae = mean_absolute_error(y_test, y_pred)
mae_capped = mean_absolute_error(y_test_capped, y_pred)

print(f"MAE (vs original):  {mae:.2f} min")
print(f"MAE (vs capped):    {mae_capped:.2f} min")

errors = np.abs(y_test - y_pred)
print(f"\nPrediction accuracy:")
for window in [5, 10, 15, 30, 60]:
    pct = (errors <= window).mean() * 100
    print(f"  Within ±{window:>2} min: {pct:.1f}%")

MAE (vs original):  18.18 min
MAE (vs capped):    16.82 min

Prediction accuracy:
  Within ± 5 min: 29.5%
  Within ±10 min: 53.4%
  Within ±15 min: 69.8%
  Within ±30 min: 88.7%
  Within ±60 min: 95.5%
